#  Reviewer Execution Guide


The intermediate sections (Section 2: Data Exploration and Section 4: Prompt Iteration & Failure Analysis) contain exploratory data analysis, debugging logs, and discarded V2 experiments included for documentation purposes, which are not required for standard pipeline execution.

In [ ]:
# Cell 1: Install the Kaggle client
!pip install -q kaggle

In [ ]:
# Cell 29 (expanded): Back up raw data folder + processed objects to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os
import pickle

DRIVE_DIR = '/content/drive/MyDrive/hiver_assignment'
os.makedirs(DRIVE_DIR, exist_ok=True)

# 1. Back up the raw data folder (so you don't have to re-download from Kaggle tomorrow)
if os.path.exists('./data') and not os.path.exists(f'{DRIVE_DIR}/data'):
    shutil.copytree('./data', f'{DRIVE_DIR}/data')
    print("Raw data folder backed up.")
else:
    print("Data folder already backed up or not found.")

# 2. Back up the processed thread objects (saves you re-running Cells 8-14 tomorrow)
with open(f'{DRIVE_DIR}/unique_threads.pkl', 'wb') as f:
    pickle.dump(unique_threads, f)
print(f"Saved {len(unique_threads)} reconstructed threads to Drive.")

# 3. Back up the current classification checkpoint
if os.path.exists('classification_results.csv'):
    shutil.copy('classification_results.csv', f'{DRIVE_DIR}/classification_results.csv')
    print("Classification checkpoint backed up.")
else:
    print("No classification checkpoint yet.")

Mounted at /content/drive
Raw data folder backed up.
Saved 10428 reconstructed threads to Drive.
Classification checkpoint backed up.


In [ ]:
# Cell 2: Set your credentials as environment variables
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input("Enter your Kaggle username: ")
os.environ['KAGGLE_KEY'] = getpass("Paste your Kaggle API key: ")

In [ ]:
# Cell 3: Download and unzip the dataset
!kaggle datasets download -d thoughtvector/customer-support-on-twitter
!unzip -q customer-support-on-twitter.zip -d ./data

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:07<00:00, 22.2MB/s]



In [ ]:
# Cell 4: Unzip the dataset-> already unzipped so rerunning -qo
!unzip -qo customer-support-on-twitter.zip -d ./data
!ls ./data

sample.csv  twcs


In [ ]:
# Cell 6: Load the full dataset into pandas
import pandas as pd

df = pd.read_csv('./data/twcs/twcs.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
# Cell 7: Find the busiest brand accounts
brand_tweets = df[df['inbound'] == False]
brand_counts = brand_tweets['author_id'].value_counts().head(20)
print(brand_counts)

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
Name: count, dtype: int64


In [ ]:
# Cell 8: Filter to AmericanAir conversations
brand_id = 'AmericanAir'

# Get all tweets FROM AmericanAir
brand_msgs = df[df['author_id'] == brand_id]

# Get the tweet_ids AmericanAir was responding TO (i.e., customer messages they replied to)
customer_tweet_ids = set(brand_msgs['in_response_to_tweet_id'].dropna())

# Get the tweet_ids that replied TO AmericanAir (i.e., customer follow-ups)
# We do this by checking which customer tweets have response_tweet_id pointing into brand_msgs' tweet_ids
brand_tweet_ids = set(brand_msgs['tweet_id'])
customer_followups = df[df['in_response_to_tweet_id'].isin(brand_tweet_ids)]

print("AmericanAir tweets:", len(brand_msgs))
print("Customer tweets AmericanAir replied to:", len(customer_tweet_ids))
print("Customer follow-up tweets:", len(customer_followups))

AmericanAir tweets: 36764
Customer tweets AmericanAir replied to: 36524
Customer follow-up tweets: 18045


In [ ]:
# Cell 9: Collect all tweet_ids relevant to AmericanAir conversations
relevant_ids = set(brand_msgs['tweet_id']) | customer_tweet_ids | set(customer_followups['tweet_id'])

# Also pull in anything those follow-ups were replying to, and anything replying to the follow-ups,
# in case threads run longer than 3 turns
relevant_ids |= set(customer_followups['in_response_to_tweet_id'].dropna())

print("Total relevant tweet_ids:", len(relevant_ids))

# Build a small, fast lookup: tweet_id -> row (only for relevant tweets, not all 2.8M)
subset_df = df[df['tweet_id'].isin(relevant_ids)]
tweet_lookup = subset_df.set_index('tweet_id').to_dict(orient='index')

print("Lookup table size:", len(tweet_lookup))

Total relevant tweet_ids: 82145
Lookup table size: 82078


In [ ]:
# Cell 10: Function to walk a thread backward from a customer follow-up to its root
def reconstruct_thread(start_tweet_id, lookup, max_turns=10):
    """
    Walks backward from start_tweet_id through in_response_to_tweet_id links,
    then reverses the result so the thread reads in chronological order.
    """
    thread = []
    current_id = start_tweet_id
    seen = set()  # guards against accidental infinite loops from bad/circular data

    for _ in range(max_turns):
        if current_id is None or pd.isna(current_id) or current_id in seen:
            break
        if current_id not in lookup:
            break  # hit one of those 67 missing/broken links — stop here

        seen.add(current_id)
        row = lookup[current_id]
        thread.append({
            'tweet_id': current_id,
            'author_id': row['author_id'],
            'inbound': row['inbound'],
            'text': row['text'],
            'created_at': row['created_at'],
        })
        current_id = row['in_response_to_tweet_id']

    thread.reverse()  # we walked backward (newest to oldest), so flip to chronological order
    return thread

In [ ]:
# Cell 11: Test on one customer follow-up
sample_id = customer_followups['tweet_id'].iloc[0]
sample_thread = reconstruct_thread(sample_id, tweet_lookup)

for turn in sample_thread:
    speaker = "CUSTOMER" if turn['inbound'] else "BRAND"
    print(f"[{speaker}] {turn['text']}\n")

[CUSTOMER] @AmericanAir Could you have someone on your lax team available to guide me to my gate ASAP

[BRAND] @115904 Our apologies for the delay in responding to you. Have you made it to LAX? Let us know if you still need assistance.

[CUSTOMER] @AmericanAir Erica on the lax team is amazing give her a raise ty



In [ ]:
# Cell 12: Reconstruct all AmericanAir conversation threads
all_threads = []

for tweet_id in customer_followups['tweet_id']:
    thread = reconstruct_thread(tweet_id, tweet_lookup)
    if len(thread) >= 2:  # keep only threads with at least one exchange (customer + brand)
        all_threads.append(thread)

print("Total reconstructed threads:", len(all_threads))
print("Example thread lengths (turns):", [len(t) for t in all_threads[:10]])

Total reconstructed threads: 18045
Example thread lengths (turns): [3, 3, 3, 5, 5, 7, 9, 9, 10, 7]


In [ ]:
# Cell 13: Check for truncated threads (hit the max_turns cap)
thread_lengths = [len(t) for t in all_threads]
truncated_count = sum(1 for l in thread_lengths if l == 10)
print("Threads that hit the max_turns cap (possibly truncated):", truncated_count)
print("Max thread length seen:", max(thread_lengths))
print("Distribution:", pd.Series(thread_lengths).describe())

Threads that hit the max_turns cap (possibly truncated): 217
Max thread length seen: 10
Distribution: count    18045.000000
mean         3.783652
std          1.527973
min          2.000000
25%          3.000000
50%          3.000000
75%          5.000000
max         10.000000
dtype: float64


In [ ]:
# Cell 12b: Rerun with a higher max_turns to avoid truncation
all_threads = []

for tweet_id in customer_followups['tweet_id']:
    thread = reconstruct_thread(tweet_id, tweet_lookup, max_turns=30)  # raised from 10 to 30
    if len(thread) >= 2:
        all_threads.append(thread)

thread_lengths = [len(t) for t in all_threads]
truncated_count = sum(1 for l in thread_lengths if l == 30)
print("Threads hitting new cap (30):", truncated_count)
print("New max length:", max(thread_lengths))

Threads hitting new cap (30): 0
New max length: 21


In [ ]:
# Cell 14: Deduplicate — keep only the longest thread per root tweet_id
dedup_map = {}

for thread in all_threads:
    root_id = thread[0]['tweet_id']
    if root_id not in dedup_map or len(thread) > len(dedup_map[root_id]):
        dedup_map[root_id] = thread

unique_threads = list(dedup_map.values())
print("Threads before dedup:", len(all_threads))
print("Threads after dedup:", len(unique_threads))

Threads before dedup: 18045
Threads after dedup: 10428


In [ ]:
# Cell 15: Sample and print 15 random conversations to read
import random
random.seed(42)  # fixes the randomness so your sample is reproducible — important, cite this in your report

sample_for_reading = random.sample(unique_threads, 15)

for i, thread in enumerate(sample_for_reading):
    print(f"--- Conversation {i+1} ---")
    for turn in thread:
        speaker = "CUSTOMER" if turn['inbound'] else "BRAND"
        print(f"[{speaker}] {turn['text']}")
    print()

--- Conversation 1 ---
[CUSTOMER] Booked flights home with @AmericanAir last night and haven't received my confirmation email and stupidly didn't write down the booking reference number. #helpmeimdumb
[BRAND] @238407 We'll gladly take a look for you, Sara. Please DM us the city pair, departure time or flight number and name as it appears on the ticket.
[CUSTOMER] @AmericanAir Appreciate your reply, got the email confirmation just as I went to write back to you &lt;3 thank you!!

--- Conversation 2 ---
[CUSTOMER] @AmericanAir nothing happening gate 50A LAX to CLT as we wait on flt attendant.  I DEMAND A REFUND!  DM Me. Now 1/2 hour boarding delay https://t.co/p6F4Maowoi
[BRAND] @158683 Meet us in DM with your record locator so we can take a peek.
[CUSTOMER] @AmericanAir where is my refund
[BRAND] @158683 If your flight is delayed an hour or more and you choose not to fly with us, you can request a refund here: https://t.co/xM8Vkc7BvM
[CUSTOMER] @AmericanAir YOUR IRRESPONSIBLE FLIGHT ATT

In [ ]:
# Cell 16: Sample a larger batch to read before finalizing the taxonomy
random.seed(42)  # same seed, but we now pull more — keeps this reproducible too

sample_for_reading_v2 = random.sample(unique_threads, 40)

for i, thread in enumerate(sample_for_reading_v2):
    print(f"--- Conversation {i+1} ---")
    for turn in thread:
        speaker = "CUSTOMER" if turn['inbound'] else "BRAND"
        print(f"[{speaker}] {turn['text']}")
    print()

--- Conversation 1 ---
[CUSTOMER] Booked flights home with @AmericanAir last night and haven't received my confirmation email and stupidly didn't write down the booking reference number. #helpmeimdumb
[BRAND] @238407 We'll gladly take a look for you, Sara. Please DM us the city pair, departure time or flight number and name as it appears on the ticket.
[CUSTOMER] @AmericanAir Appreciate your reply, got the email confirmation just as I went to write back to you &lt;3 thank you!!

--- Conversation 2 ---
[CUSTOMER] @AmericanAir nothing happening gate 50A LAX to CLT as we wait on flt attendant.  I DEMAND A REFUND!  DM Me. Now 1/2 hour boarding delay https://t.co/p6F4Maowoi
[BRAND] @158683 Meet us in DM with your record locator so we can take a peek.
[CUSTOMER] @AmericanAir where is my refund
[BRAND] @158683 If your flight is delayed an hour or more and you choose not to fly with us, you can request a refund here: https://t.co/xM8Vkc7BvM
[CUSTOMER] @AmericanAir YOUR IRRESPONSIBLE FLIGHT ATT

In [ ]:
# Cell 17: Install Groq client
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.3 MB/s eta 0:00:00


In [ ]:
# Cell 18: Set your Groq API key
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass("Paste your Groq API key: ")

from groq import Groq
client = Groq()  # automatically reads GROQ_API_KEY from the environment

In [ ]:
# Cell 19: Intent taxonomy, with clear definitions for the LLM to use
INTENT_TAXONOMY = {
    "flight_disruption_refund": "Delays, cancellations, missed connections, or requests for refunds/compensation due to a disrupted flight.",
    "booking_reservation": "Issues with booking confirmations, reservation references, name changes, or reservation modifications.",
    "baggage_fees": "Questions or complaints about baggage handling, checked-bag fees, or other ancillary fees.",
    "policy_information": "General questions about airline policy, rules, or pricing that are not tied to a specific disrupted trip (e.g., carry-on rules, upgrade eligibility).",
    "service_quality_complaint": "Complaints about staff behavior, rudeness, poor treatment, or onboard discomfort, often seeking acknowledgment or compensation.",
    "technical_app_issue": "Problems with the airline's app, website, or digital check-in/booking systems.",
    "positive_feedback": "Compliments, gratitude, or positive comments with no actionable request.",
    "non_actionable_other": "Sarcasm, off-topic content, out-of-scope requests, or abusive/toxic language not requiring a substantive support response.",
}

In [ ]:
# Cell 20: Format a thread as readable text for the prompt
def format_thread(thread):
    lines = []
    for turn in thread:
        speaker = "Customer" if turn['inbound'] else "Brand"
        lines.append(f"{speaker}: {turn['text']}")
    return "\n".join(lines)

In [ ]:
# Cell 21: Build the classification prompt
def build_classification_prompt(thread):
    taxonomy_text = "\n".join([f"- {key}: {desc}" for key, desc in INTENT_TAXONOMY.items()])
    thread_text = format_thread(thread)

    prompt = f"""You are classifying customer support conversations for an airline (American Airlines) on Twitter.

Classify the CUSTOMER'S PRIMARY INTENT in this conversation into exactly one of these categories:

{taxonomy_text}

Conversation:
{thread_text}

Respond with ONLY the category key (e.g., "flight_disruption_refund") and nothing else. No explanation, no punctuation."""
    return prompt

In [ ]:
# Cell 22 (fixed): Classify with explicit model + reasoning disabled
def classify_intent(thread, model="openai/gpt-oss-20b"):
    prompt = build_classification_prompt(thread)
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=200,          # generous headroom in case reasoning can't be fully disabled
            reasoning_effort="low",  # gpt-oss-specific param: minimizes the hidden reasoning trace
        )
        label = response.choices[0].message.content.strip()
        if label not in INTENT_TAXONOMY:
            return "PARSE_ERROR", label
        return label, None
    except Exception as e:
        return "API_ERROR", str(e)

In [ ]:
# Cell 23: Test on 3 conversations
for i, thread in enumerate(unique_threads[:3]):
    label, error = classify_intent(thread)
    print(f"--- Thread {i+1} ---")
    print(format_thread(thread)[:200], "...")
    print(f"Predicted intent: {label}")
    if error:
        print(f"Error/raw output: {error}")
    print()

--- Thread 1 ---
Customer: @AmericanAir Could you have someone on your lax team available to guide me to my gate ASAP
Brand: @115904 Our apologies for the delay in responding to you. Have you made it to LAX? Let us kn ...
Predicted intent: positive_feedback

--- Thread 2 ---
Customer: I’m sorry, what? It’s going to COST me $50 to transfer 4,000 AA Advantage points to my spouse? @AmericanAir this is ridiculous!!
Brand: @115906 This is a great option for customers who want  ...
Predicted intent: policy_information

--- Thread 3 ---
Customer: Trying to book a flight on @AmericanAir and an error comes up: System having trouble. Anyone else having problem?
Brand: @116144 We're sorry for any difficulties that you've experienced. Ple ...
Predicted intent: booking_reservation



In [ ]:
# Cell 26: Sample threads for classification
random.seed(42)  # same seed as before, for reproducibility
SAMPLE_SIZE = 1200
classification_sample = random.sample(unique_threads, SAMPLE_SIZE)
print(f"Sampling {len(classification_sample)} threads out of {len(unique_threads)} unique conversations")

Sampling 1200 threads out of 10428 unique conversations


In [ ]:
# Cell 27: Install tqdm for a progress bar
!pip install -q tqdm

In [ ]:
# Cell 28: Run classification with checkpointing
import time
import csv
from tqdm import tqdm

CHECKPOINT_FILE = 'classification_results.csv'
results = []

# Resume support: if a checkpoint already exists, load it so a rerun doesn't start from scratch
try:
    existing = pd.read_csv(CHECKPOINT_FILE)
    results = existing.to_dict('records')
    already_done = set(existing['root_tweet_id'])
    print(f"Resuming: {len(already_done)} threads already classified")
except FileNotFoundError:
    already_done = set()
    print("No checkpoint found, starting fresh")

for thread in tqdm(classification_sample):
    root_id = thread[0]['tweet_id']
    if root_id in already_done:
        continue  # skip threads we've already classified in a previous run

    label, error = classify_intent(thread)

    results.append({
        'root_tweet_id': root_id,
        'thread_text': format_thread(thread),
        'num_turns': len(thread),
        'predicted_intent': label,
        'error': error,
    })

    # Save every 50 calls, so a crash/disconnect loses at most 50 calls of progress
    if len(results) % 50 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)

    time.sleep(0.3)  # small delay to stay comfortably under free-tier rate limits

# Final save
pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
print(f"Done. Total classified: {len(results)}")

No checkpoint found, starting fresh


100%|██████████| 1200/1200 [1:01:43<00:00,  3.09s/it]

Done. Total classified: 1200


In [ ]:
print(pd.DataFrame(results)['predicted_intent'].value_counts())
print(pd.DataFrame(results)['error'].value_counts())

NameError: name 'pd' is not defined

In [ ]:
# Tomorrow's Cell 1: Restore from Drive instead of rebuilding
from google.colab import drive
drive.mount('/content/drive')

import pickle

import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/hiver_assignment'

with open(f'{DRIVE_DIR}/unique_threads.pkl', 'rb') as f:
    unique_threads = pickle.load(f)
print(f"Restored {len(unique_threads)} threads.")

# Resume classification checkpoint if it exists
try:
    progress = pd.read_csv(f'{DRIVE_DIR}/classification_results.csv')
    print(f"Classification progress: {len(progress)} threads already done.")
except FileNotFoundError:
    print("No classification progress found yet.")

Mounted at /content/drive
Restored 10428 threads.
Classification progress: 1200 threads already done.


In [ ]:
# Cell 30: Load and analyze classification results
import pandas as pd

results_df = pd.read_csv('/content/drive/MyDrive/hiver_assignment/classification_results.csv')

print("Total classified:", len(results_df))
print("\n--- Intent distribution ---")
print(results_df['predicted_intent'].value_counts())
print("\n--- Error breakdown ---")
print(results_df['error'].value_counts(dropna=False))

Total classified: 1200

--- Intent distribution ---
predicted_intent
API_ERROR                    438
flight_disruption_refund     253
service_quality_complaint    177
policy_information            91
positive_feedback             66
baggage_fees                  62
non_actionable_other          59
booking_reservation           42
technical_app_issue           12
Name: count, dtype: int64

--- Error breakdown ---
error
NaN                                                                                                                                                                                                                                                                                                                                                                                                                          762
Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01jpejp8cder090g4h74xrh98m` service tier `on_dem

Hit Groq's free-tier daily token quota (200,000 TPD) after 762/1200 classification calls.
Rather than wait ~24h for reset or pay for Dev Tier, proceeded with 762 successful
classifications — well above the 150-250 needed for the golden evaluation set.
This is a real-world constraint an actual production system would need to budget/plan
around (token costs, rate limits, provider quotas).

In [ ]:
# Cell 31: Filter to only successful classifications, save a clean working set
clean_results = results_df[results_df['predicted_intent'] != 'API_ERROR'].copy()
print(f"Working dataset: {len(clean_results)} successfully classified threads")
print(clean_results['predicted_intent'].value_counts())

Working dataset: 762 successfully classified threads
predicted_intent
flight_disruption_refund     253
service_quality_complaint    177
policy_information            91
positive_feedback             66
baggage_fees                  62
non_actionable_other          59
booking_reservation           42
technical_app_issue           12
Name: count, dtype: int64


In [ ]:
# Cell 32: Stratified sample for golden set labeling
GOLDEN_SET_SIZE = 200
TARGET_PER_CLASS = GOLDEN_SET_SIZE // len(clean_results['predicted_intent'].unique())  # even split target

golden_candidates = []
for intent, group in clean_results.groupby('predicted_intent'):
    n = min(TARGET_PER_CLASS, len(group))  # can't oversample beyond what exists (e.g., technical_app_issue has only 12)
    sampled = group.sample(n=n, random_state=42)
    golden_candidates.append(sampled)

golden_df = pd.concat(golden_candidates).reset_index(drop=True)
print(f"Golden set candidate pool: {len(golden_df)}")
print(golden_df['predicted_intent'].value_counts())

Golden set candidate pool: 187
predicted_intent
baggage_fees                 25
booking_reservation          25
flight_disruption_refund     25
non_actionable_other         25
policy_information           25
positive_feedback            25
service_quality_complaint    25
technical_app_issue          12
Name: count, dtype: int64


In [ ]:
# Cell 33: Prepare a blind labeling sheet
labeling_df = golden_df[['root_tweet_id', 'thread_text']].copy()
labeling_df['human_label'] = ''  # empty column for you to fill in
labeling_df = labeling_df.sample(frac=1, random_state=7).reset_index(drop=True)  # shuffle so category order doesn't bias you

labeling_df.to_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv', index=False)
print(f"Saved {len(labeling_df)} conversations for blind labeling.")
print("\nValid intent keys to use:")
for key in INTENT_TAXONOMY:
    print(f" - {key}")

Saved 187 conversations for blind labeling.

Valid intent keys to use:
 - flight_disruption_refund
 - booking_reservation
 - baggage_fees
 - policy_information
 - service_quality_complaint
 - technical_app_issue
 - positive_feedback
 - non_actionable_other


In [ ]:
# Cell 34: Batch labeling — 5 conversations at a time
labeling_df = pd.read_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv')
valid_keys = list(INTENT_TAXONOMY.keys())
BATCH_SIZE = 5

# Print the full taxonomy with definitions once, as a reference while you label
print("="*60)
print("INTENT TAXONOMY REFERENCE")
print("="*60)
for key, desc in INTENT_TAXONOMY.items():
    print(f"\n{key}:\n  {desc}")
print("\n" + "="*60 + "\n")

unlabeled_idx = labeling_df[labeling_df['human_label'].isna() | (labeling_df['human_label'] == '')].index.tolist()

for batch_start in range(0, len(unlabeled_idx), BATCH_SIZE):
    batch_idx = unlabeled_idx[batch_start:batch_start + BATCH_SIZE]

    for pos, i in enumerate(batch_idx):
        print(f"\n[{pos+1}] (row {i}, overall {i+1}/{len(labeling_df)})")
        print(labeling_df.loc[i, 'thread_text'])

    print("\nOptions:", ", ".join(valid_keys))
    print(f"\nEnter {len(batch_idx)} labels separated by commas, in order (e.g. flight_disruption_refund, baggage_fees, ...):")
    raw = input("Labels: ").strip()
    labels = [l.strip() for l in raw.split(',')]

    while len(labels) != len(batch_idx) or any(l not in valid_keys for l in labels):
        print("Mismatch or invalid key found. Please retype all labels for this batch.")
        raw = input("Labels: ").strip()
        labels = [l.strip() for l in raw.split(',')]

    for i, label in zip(batch_idx, labels):
        labeling_df.at[i, 'human_label'] = label

    labeling_df.to_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv', index=False)
    print(f"Saved. Progress: {batch_start + len(batch_idx)}/{len(unlabeled_idx)}")

print("Labeling complete!")

INTENT TAXONOMY REFERENCE

flight_disruption_refund:
  Delays, cancellations, missed connections, or requests for refunds/compensation due to a disrupted flight.

booking_reservation:
  Issues with booking confirmations, reservation references, name changes, or reservation modifications.

baggage_fees:
  Questions or complaints about baggage handling, checked-bag fees, or other ancillary fees.

policy_information:
  General questions about airline policy, rules, or pricing that are not tied to a specific disrupted trip (e.g., carry-on rules, upgrade eligibility).

service_quality_complaint:
  Complaints about staff behavior, rudeness, poor treatment, or onboard discomfort, often seeking acknowledgment or compensation.

technical_app_issue:
  Problems with the airline's app, website, or digital check-in/booking systems.

positive_feedback:
  Compliments, gratitude, or positive comments with no actionable request.

non_actionable_other:
  Sarcasm, off-topic content, out-of-scope requests

/tmp/ipykernel_2284/492733851.py:34: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'flight_disruption_refund' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  labeling_df.at[i, 'human_label'] = label


Saved. Progress: 5/187

[1] (row 5, overall 6/187)
Customer: @AmericanAir How do you get this back to your car without a handle or anything to be able to zip it up? Carry the greasy bag on your back
Brand: @341969 Please work with our team at the airport and they'll provide you with as much help possible.
Customer: @AmericanAir Issue resolved - thanks to a lovely customer service agent Sharon (50626). Please find her and give her a kudos
Brand: @341969 We'd love to pass your praises along! Was Sharon an airport agent or did you speak with her over the phone? #AATeam
Customer: @AmericanAir Spoke with her over the phone - she walked me through the options and next steps and we resolved the issue this morning.

[2] (row 6, overall 7/187)
Customer: I honestly have never had a worse experience than this evening with @AmericanAir, and that includes a 12-Hour flight delay with Southwest I once had. AA is an embarrassment &amp; the AA staff @150414 are truly awful. Honestly this has to be a jo

In [ ]:
labeling_df.to_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv', index=False)

In [ ]:
# Cell 35: Verify labeling is saved and complete
saved_df = pd.read_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv')
print(f"Total rows: {len(saved_df)}")
print(f"Labeled rows: {saved_df['human_label'].notna().sum()}")
print(f"Unlabeled rows: {saved_df['human_label'].isna().sum()}")
print("\nYour label distribution:")
print(saved_df['human_label'].value_counts())

Total rows: 187
Labeled rows: 187
Unlabeled rows: 0

Your label distribution:
human_label
baggage_fees                 35
flight_disruption_refund     34
positive_feedback            32
booking_reservation          29
service_quality_complaint    26
policy_information           18
technical_app_issue          12
non_actionable_other          1
Name: count, dtype: int64


In [ ]:
# Cell 36: Merge human labels with model predictions and compute real accuracy
human_labels = pd.read_csv('/content/drive/MyDrive/hiver_assignment/golden_set_blind.csv')[['root_tweet_id', 'human_label']]

# golden_df still holds the model's original predictions for these same 187 conversations
comparison_df = golden_df.merge(human_labels, on='root_tweet_id', how='inner')

comparison_df['agree'] = comparison_df['predicted_intent'] == comparison_df['human_label']

accuracy = comparison_df['agree'].mean()
print(f"Overall agreement (LLM accuracy vs. human ground truth): {accuracy:.1%}")
print(f"Total compared: {len(comparison_df)}")

Overall agreement (LLM accuracy vs. human ground truth): 69.0%
Total compared: 187


In [ ]:
# Cell 37: Per-category accuracy (much more informative than one overall number)
per_class_accuracy = comparison_df.groupby('predicted_intent')['agree'].agg(['mean', 'count'])
per_class_accuracy.columns = ['accuracy', 'n_samples']
print(per_class_accuracy.sort_values('accuracy'))

                           accuracy  n_samples
predicted_intent                              
non_actionable_other       0.040000         25
policy_information         0.520000         25
service_quality_complaint  0.640000         25
booking_reservation        0.760000         25
technical_app_issue        0.833333         12
positive_feedback          0.880000         25
flight_disruption_refund   0.960000         25
baggage_fees               0.960000         25


In [ ]:
# Cell 38: Full confusion matrix - what the model predicted vs. what it actually was
confusion = pd.crosstab(comparison_df['predicted_intent'], comparison_df['human_label'], margins=True)
print(confusion)

human_label                baggage_fees  booking_reservation  \
predicted_intent                                               
baggage_fees                         24                    0   
booking_reservation                   0                   19   
flight_disruption_refund              0                    0   
non_actionable_other                  2                    1   
policy_information                    5                    4   
positive_feedback                     1                    2   
service_quality_complaint             3                    2   
technical_app_issue                   0                    1   
All                                  35                   29   

human_label                flight_disruption_refund  non_actionable_other  \
predicted_intent                                                            
baggage_fees                                      0                     0   
booking_reservation                               2             

My 69% overall accuracy is misleading because it averages across categories with
wildly different reliability (96% on baggage_fees/flight_disruption_refund vs. 4% on
non_actionable_other). A stakeholder reading "69% accurate" would not realize the system
is essentially non-functional for one entire intent category. Because my golden set was
balanced (~25 per class), the aggregate number happens to be close to the macro-average
across classes — but it still conceals that failures are heavily concentrated, not evenly
spread. In production, this matters enormously for the escalation decision: if
non_actionable_other conversations are actually service complaints 28% of the time,
routing them as "no action needed" risks silently dropping real customer complaints.

In [ ]:
# Cell 39: Trivial baseline - always predict the most common human label
majority_class = comparison_df['human_label'].value_counts().idxmax()
trivial_accuracy = (comparison_df['human_label'] == majority_class).mean()

print(f"Majority class: {majority_class}")
print(f"Trivial baseline accuracy: {trivial_accuracy:.1%}")

Majority class: baggage_fees
Trivial baseline accuracy: 18.7%


In [ ]:
# Cell 40: Simple keyword-based classifier
def keyword_classify(text):
    text_lower = text.lower()

    # Ordered rules - first match wins, order reflects specificity (most specific first)
    if any(w in text_lower for w in ['refund', 'delay', 'delayed', 'cancel', 'cancelled', 'compensation for delay']):
        return 'flight_disruption_refund'
    if any(w in text_lower for w in ['bag', 'baggage', 'luggage', 'checked bag', 'carry-on', 'carry on']):
        return 'baggage_fees'
    if any(w in text_lower for w in ['confirmation', 'booking', 'reservation', 'record locator', 'itinerary']):
        return 'booking_reservation'
    if any(w in text_lower for w in ['app', 'website', 'error', 'system', 'check-in', 'login']):
        return 'technical_app_issue'
    if any(w in text_lower for w in ['rude', 'terrible', 'worst', 'poor service', 'compensation', 'complain']):
        return 'service_quality_complaint'
    if any(w in text_lower for w in ['thank', 'thanks', 'love', 'great', 'amazing', 'awesome']):
        return 'positive_feedback'
    if any(w in text_lower for w in ['policy', 'allowed', 'can i', 'rule', 'how much', 'cost']):
        return 'policy_information'

    return 'non_actionable_other'  # default fallback if nothing matches

# Apply to the golden set
comparison_df['simple_baseline_pred'] = comparison_df['thread_text'].apply(keyword_classify)
comparison_df['simple_agree'] = comparison_df['simple_baseline_pred'] == comparison_df['human_label']

simple_accuracy = comparison_df['simple_agree'].mean()
print(f"Simple (keyword) baseline accuracy: {simple_accuracy:.1%}")

print("\nPer-category accuracy:")
print(comparison_df.groupby('human_label')['simple_agree'].agg(['mean', 'count']))

Simple (keyword) baseline accuracy: 49.7%

Per-category accuracy:
                               mean  count
human_label                               
baggage_fees               0.828571     35
booking_reservation        0.517241     29
flight_disruption_refund   0.470588     34
non_actionable_other       1.000000      1
policy_information         0.000000     18
positive_feedback          0.562500     32
service_quality_complaint  0.076923     26
technical_app_issue        1.000000     12


In [ ]:
# Cell 41: Three-way comparison
print(f"{'Method':<20} {'Accuracy':<10}")
print(f"{'Trivial (majority)':<20} {trivial_accuracy:.1%}")
print(f"{'Simple (keyword)':<20} {simple_accuracy:.1%}")
print(f"{'LLM (gpt-oss-20b)':<20} {accuracy:.1%}")

Method               Accuracy  
Trivial (majority)   18.7%
Simple (keyword)     49.7%
LLM (gpt-oss-20b)    69.0%


The keyword baseline's per-category breakdown reveals a structural weakness beyond
just "keywords are less smart than an LLM": rigid, ordered if-else matching means a
service complaint mentioning "delayed" gets captured by the flight_disruption_refund
rule before ever reaching service_quality_complaint (7.7% accuracy) — the baseline
fails not from missing vocabulary, but from an inability to weigh competing signals
within one message, which is exactly the kind of judgment an LLM handles natively.

In [ ]:
# Cell 42: Revised taxonomy with a tightened non_actionable_other definition
INTENT_TAXONOMY_V2 = INTENT_TAXONOMY.copy()

INTENT_TAXONOMY_V2["non_actionable_other"] = (
    "ONLY use this for content with no discernible customer service need at all: "
    "pure sarcasm/trolling with no real request, off-topic content unrelated to the airline, "
    "or abusive language with no actionable complaint. "
    "Do NOT use this if the customer is complaining about treatment/service (use service_quality_complaint instead), "
    "expressing genuine gratitude or praise (use positive_feedback instead), "
    "or mentioning a delay/cancellation even briefly (use flight_disruption_refund instead). "
    "When in doubt between this category and any other, choose the other category."
)

print(INTENT_TAXONOMY_V2["non_actionable_other"])

ONLY use this for content with no discernible customer service need at all: pure sarcasm/trolling with no real request, off-topic content unrelated to the airline, or abusive language with no actionable complaint. Do NOT use this if the customer is complaining about treatment/service (use service_quality_complaint instead), expressing genuine gratitude or praise (use positive_feedback instead), or mentioning a delay/cancellation even briefly (use flight_disruption_refund instead). When in doubt between this category and any other, choose the other category.



The full journey: 36,764 → 187

Step 1 — Raw data (2,811,774 total tweets in the dataset). Filtered to AmericanAir: found 36,764 tweets sent by AmericanAir, which led to 36,524 customer tweets they replied to, and 18,045 customer follow-up tweets.

Step 2 — Thread reconstruction. The dataset only stores tweet-to-tweet reply links (response_tweet_id/in_response_to_tweet_id), not full conversations. You wrote a function to walk that chain and rebuild full multi-turn threads.

Failure #1: First attempt used max_turns=10, and 217 threads (1.2%) hit that cap — meaning real conversations were being cut off mid-thread.
Fix: Raised the cap to 30; confirmed real max thread length was 21, so nothing was truncated anymore.
Failure #2: Building one thread per follow-up tweet produced heavy duplication — 18,045 raw threads for what were really far fewer unique conversations (a 4-turn conversation with 2 follow-ups gets reconstructed twice).
Fix: Deduplicated by keeping only the longest thread per root tweet_id → 10,428 unique conversations.

Step 3 — Intent taxonomy, built from real data, not guessed. Read 55 real conversations (15, then 40 more) and derived 8 categories from actual observed patterns: flight disruption/refund, booking/reservation, baggage/fees, policy/info, service complaint, technical/app issue, positive feedback, and non-actionable/other.

Step 4 — LLM classification at scale.

Failure #3: Your first model choice, llama-3.1-8b-instant, turned out to be deprecated by Groq. The API silently substituted a different model instead of erroring, which caused confusing behavior.
Failure #4: The substitute model (openai/gpt-oss-20b) is a reasoning model — your max_tokens=20 was too small, so all tokens got consumed by hidden reasoning, leaving empty responses (PARSE_ERROR on everything).
Fix: Pinned the model explicitly, raised max_tokens to 200, added reasoning_effort="low".
Failure #5: Ran into Groq's free-tier daily token quota (200,000 tokens/day) partway through — 438 of 1,200 calls failed with 429 errors.
Fix: Didn't wait for reset — proceeded with the 762 successful classifications, since that's well above the 150-250 needed. Also mitigated risk by mounting Google Drive for checkpointing, so progress wasn't lost to Colab session limits.

Step 5 — Golden set construction. From the 762 successful classifications, built a stratified sample (not pure random) so rare categories weren't drowned out — landed on 187 candidates (capped by technical_app_issue only having 12 total examples available).

Step 6 — Blind human labeling. You personally read and labeled all 187 conversations without seeing the model's predicted label, to avoid bias — this is your independent ground truth.

Performance results
Method	Accuracy
Trivial baseline (always guess most common class)	18.7%
Simple baseline (keyword rules)	49.7%
LLM classifier (your system)	69.0%

The critical finding — the 69% hides a major problem:

baggage_fees and flight_disruption_refund: 96% each — excellent.
non_actionable_other: 4% — essentially broken. The model uses this as a dumping ground for anything unclear, mainly confusing it with service_quality_complaint (7 cases) and positive_feedback (6 cases).
This matters beyond accuracy: if a real complaint gets mislabeled as "non-actionable," it could get wrongly auto-closed instead of escalated — a real product risk, not just a metrics footnote.
Also found: your keyword baseline scored 0% on policy_information and 7.7% on service_quality_complaint — not from missing vocabulary, but because earlier rules (checked first) swallow those cases before they're ever reached. A precise, evidence-backed reason the LLM approach earns its complexity.
What's left to build
Retrieval-grounded reply drafting (biggest remaining piece — hasn't started)
Escalation decision logic — now informed by real evidence (e.g., treat non_actionable_other predictions with extra caution given its 4% reliability)
LLM-as-judge for reply quality + judge-vs-human agreement validation


In [ ]:
# Cell 43: Rebuild prompt builder with V2 taxonomy
def build_classification_prompt_v2(thread):
    taxonomy_text = "\n".join([f"- {key}: {desc}" for key, desc in INTENT_TAXONOMY_V2.items()])
    thread_text = format_thread(thread)

    prompt = f"""You are classifying customer support conversations for an airline (American Airlines) on Twitter.

Classify the CUSTOMER'S PRIMARY INTENT in this conversation into exactly one of these categories:

{taxonomy_text}

Conversation:
{thread_text}

Respond with ONLY the category key (e.g., "flight_disruption_refund") and nothing else. No explanation, no punctuation."""
    return prompt

def classify_intent_v2(thread, model="openai/gpt-oss-20b"):
    prompt = build_classification_prompt_v2(thread)
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=200,
            reasoning_effort="low",
        )
        label = response.choices[0].message.content.strip()
        if label not in INTENT_TAXONOMY_V2:
            return "PARSE_ERROR", label
        return label, None
    except Exception as e:
        return "API_ERROR", str(e)

In [ ]:
# Cell 44: Re-classify only the 187 golden-set threads with the improved prompt
import time

v2_results = []
for _, row in comparison_df.iterrows():
    thread_text = row['thread_text']
    fake_thread = [{'inbound': True, 'text': thread_text}]  # reuse text directly, see note below
    label, error = classify_intent_v2(fake_thread)
    v2_results.append({'root_tweet_id': row['root_tweet_id'], 'v2_predicted': label, 'error': error})
    time.sleep(0.3)

v2_df = pd.DataFrame(v2_results)
print(v2_df['v2_predicted'].value_counts())
print(v2_df['error'].value_counts(dropna=False))

v2_predicted
service_quality_complaint    35
flight_disruption_refund     28
booking_reservation          26
baggage_fees                 25
policy_information           20
positive_feedback            20
non_actionable_other         18
technical_app_issue          12
PARSE_ERROR                   3
Name: count, dtype: int64
error
None    184
          3
Name: count, dtype: int64


In [ ]:
# Cell 45: Scoring-based keyword classifier instead of first-match ordering
def keyword_classify_v2(text):
    text_lower = text.lower()

    keyword_sets = {
        'flight_disruption_refund': ['refund', 'delayed', 'delay', 'cancelled', 'cancel', 'missed connection'],
        'baggage_fees': ['baggage', 'checked bag', 'carry-on', 'carry on', 'luggage'],
        'booking_reservation': ['confirmation', 'booking', 'reservation', 'record locator', 'itinerary'],
        'technical_app_issue': ['app', 'website', 'system error', 'check-in issue', 'login'],
        'service_quality_complaint': ['rude', 'terrible', 'worst', 'poor service', 'compensation', 'unacceptable', 'incompetence'],
        'positive_feedback': ['thank you', 'thanks', 'love', 'amazing', 'awesome', 'great job'],
        'policy_information': ['policy', 'allowed', 'how much', 'what is the', 'rule'],
    }

    scores = {intent: sum(1 for kw in kws if kw in text_lower) for intent, kws in keyword_sets.items()}
    best_intent = max(scores, key=scores.get)

    if scores[best_intent] == 0:
        return 'non_actionable_other'
    return best_intent

comparison_df['simple_v2_pred'] = comparison_df['thread_text'].apply(keyword_classify_v2)
comparison_df['simple_v2_agree'] = comparison_df['simple_v2_pred'] == comparison_df['human_label']
print(f"Simple baseline V2 accuracy: {comparison_df['simple_v2_agree'].mean():.1%}")
print(comparison_df.groupby('human_label')['simple_v2_agree'].agg(['mean', 'count']))

Simple baseline V2 accuracy: 42.2%
                               mean  count
human_label                               
baggage_fees               0.571429     35
booking_reservation        0.448276     29
flight_disruption_refund   0.411765     34
non_actionable_other       1.000000      1
policy_information         0.000000     18
positive_feedback          0.593750     32
service_quality_complaint  0.115385     26
technical_app_issue        0.750000     12


In [ ]:
# Cell 46: Merge V2 predictions with human labels and compute accuracy
v2_comparison = comparison_df.merge(v2_df, on='root_tweet_id', how='inner')
v2_comparison['v2_agree'] = v2_comparison['v2_predicted'] == v2_comparison['human_label']

v2_accuracy = v2_comparison['v2_agree'].mean()
print(f"V1 (original) accuracy: {accuracy:.1%}")
print(f"V2 (improved prompt) accuracy: {v2_accuracy:.1%}")

print("\nPer-category accuracy, V1 vs V2:")
v1_per_class = comparison_df.groupby('human_label')['agree'].mean()
v2_per_class = v2_comparison.groupby('human_label')['v2_agree'].mean()
side_by_side = pd.DataFrame({'v1_accuracy': v1_per_class, 'v2_accuracy': v2_per_class})
side_by_side['improvement'] = side_by_side['v2_accuracy'] - side_by_side['v1_accuracy']
print(side_by_side.sort_values('improvement', ascending=False))

V1 (original) accuracy: 69.0%
V2 (improved prompt) accuracy: 66.8%

Per-category accuracy, V1 vs V2:
                           v1_accuracy  v2_accuracy  improvement
human_label                                                     
service_quality_complaint     0.615385     0.692308     0.076923
flight_disruption_refund      0.705882     0.764706     0.058824
booking_reservation           0.655172     0.689655     0.034483
technical_app_issue           0.833333     0.833333     0.000000
baggage_fees                  0.685714     0.657143    -0.028571
positive_feedback             0.687500     0.593750    -0.093750
policy_information            0.722222     0.500000    -0.222222
non_actionable_other          1.000000     0.000000    -1.000000


A targeted prompt fix for non_actionable_other's over-prediction succeeded narrowly
(reduced its usage from 25→18 predictions) but caused a net regression in overall
accuracy (69.0%→66.8%), with recall dropping on policy_information (-22.2%) and
positive_feedback (-9.4%). This demonstrates that the model's uncertainty on
ambiguous conversations doesn't disappear when a category is discouraged — it
gets redistributed to other categories, sometimes incorrectly. A more robust fix
would need few-shot examples per category rather than definition tweaks alone,
which I'd pursue with more time (see "next steps" section).




## Why the accuracy went down despite fixing the target problem

Reducing over-prediction of a bad category doesn't automatically raise accuracy — it only helps *if* the reassigned predictions land correctly. Look at what actually happened, category by category:

**Where the fix worked as intended:**
- `service_quality_complaint`: +7.7% — this was the single biggest leak destination in V1 (7 cases wrongly called `non_actionable_other`). The fix correctly recovered some of these.
- `flight_disruption_refund`: +5.9% — same story, this was the second-biggest leak (5 cases).

**Where it backfired:**
- `policy_information`: **-22.2%** — this is the real problem. Only 2 policy conversations were originally leaking into `non_actionable_other` in V1 — nowhere near enough to explain a 22% drop. What likely happened: your new instruction ("when in doubt, choose the other category") didn't just fix the specific leak you diagnosed — it made the model **more trigger-happy about avoiding `non_actionable_other` in general**, which pushed some conversations that were *previously correctly classified* as `policy_information` into some *other* wrong category instead.
- `positive_feedback`: -9.4%, `baggage_fees`: -2.9% — smaller versions of the same side effect.

**The core lesson:** you told the model "avoid Category X," but the model doesn't have surgical precision — it can't distinguish "avoid X only in the exact situations that were leaking" from "generally avoid X more." A blunt instruction fixed the specific leak but created new noise elsewhere. This is a completely normal, well-known failure mode in prompt engineering (and in ML more broadly) — you can't tune away one error mode without carefully checking you haven't introduced others, which is exactly why you re-tested on the full golden set instead of just eyeballing the `non_actionable_other` count.



That last step is the important one. A lot of people would have shipped V2 without re-testing on the golden set, just trusting that "the count went down, so it must be better."



> "Diagnosed that `non_actionable_other` was being over-used as a catch-all, confirmed it with a confusion matrix, and tested a targeted prompt fix. It worked for the specific leak I found — recovered accuracy in `service_quality_complaint` and `flight_disruption_refund` — but caused a net regression overall, because the instruction made the model more conservative about that category in ways that hurt unrelated classes like `policy_information`. Rather than keep iterating on the prompt blind, I kept the better-performing V1 classifier and addressed the known weakness at the system level instead — routing low-confidence categories through stricter escalation rather than relying on a perfect classifier. Given more time, I'd explore few-shot examples instead of definition tweaks, which are usually more surgical than instruction-based fixes."



In [ ]:
# Cell 48: Filter to threads with a substantive brand response
def has_substantive_reply(thread):
    # A resolved/useful thread needs at least one brand turn with reasonable length
    # (filters out low-content replies like "We see your DM" or single emoji responses)
    brand_turns = [t for t in thread if not t['inbound']]
    return any(len(t['text'].split()) >= 8 for t in brand_turns)

resolvable_threads = [t for t in unique_threads if has_substantive_reply(t)]
print(f"Threads with substantive brand response: {len(resolvable_threads)} / {len(unique_threads)}")

Threads with substantive brand response: 10286 / 10428


In [ ]:
# Cell 49: Install sentence-transformers and FAISS
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.7 MB/s eta 0:00:00


In [ ]:
# Cell 50: Embed all resolvable threads
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# We embed just the CUSTOMER's opening message - that's what a new incoming message will be compared against
customer_openers = [thread[0]['text'] for thread in resolvable_threads]

embeddings = embed_model.encode(customer_openers, show_progress_bar=True, batch_size=64)
print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Embeddings shape: (10286, 384)


In [ ]:
# Cell 51: Build a FAISS index for fast similarity search
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product - works well with normalized embeddings for cosine similarity

# Normalize embeddings so inner product behaves like cosine similarity
faiss.normalize_L2(embeddings)
index.add(embeddings.astype('float32'))

print(f"Index built with {index.ntotal} vectors")

Index built with 10286 vectors


In [ ]:
# Cell 52: Test retrieval with a new, made-up incoming message
def retrieve_similar(query_text, k=3):
    query_vec = embed_model.encode([query_text])
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec.astype('float32'), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'score': float(score),
            'thread': resolvable_threads[idx],
        })
    return results

test_query = "My flight got delayed 3 hours and I want a refund"
results = retrieve_similar(test_query, k=3)

for r in results:
    print(f"\nSimilarity: {r['score']:.3f}")
    print(format_thread(r['thread']))


Similarity: 0.894
Customer: my flight is now delayed until tomorrow at 9:30am 

@americanair i want a refund
Brand: @366317 We did have a crew for your flight however, our crew can be displaced due to operational reasons. Thanks for your patience tonight.
Customer: @AmericanAir i asked for a refund not excuses

Similarity: 0.684
Customer: @AmericanAir it’s ridiculous that after spending hundreds of dollars on a ticket they reufuse to issue a refund because it’s “too late”. I was traveling internationally and was only able to contact customer service 48hrs after my flight. 😡😡😡😡 https://t.co/cAT2gr8Wh8
Brand: @763070 We're very sorry this happened. Please make sure you file a claim with our Bag team before you leave the airport.
Customer: @AmericanAir I filed a claim online and they just replied this morning saying that it was too late. I had an international flight afterwards and had no time to file a claim in the airport. The flight was on the 15th and I submitted the claim the 19th



In [ ]:
# Cell 53: Reply drafting prompt, grounded in retrieved similar resolutions
def build_reply_prompt(new_message, retrieved_examples):
    examples_text = ""
    for i, ex in enumerate(retrieved_examples):
        examples_text += f"\n--- Similar past conversation {i+1} (similarity: {ex['score']:.2f}) ---\n"
        examples_text += format_thread(ex['thread'])
        examples_text += "\n"

    prompt = f"""You are a customer support agent for American Airlines, responding on Twitter.

Here are similar past conversations showing how American Airlines has historically responded to similar issues:
{examples_text}

Now draft a reply to this NEW customer message, in American Airlines' typical tone and style
(empathetic, brief, action-oriented, often asks for a DM/record locator for account-specific issues).
Do not invent specific facts (flight numbers, compensation amounts, policy details) that aren't
grounded in the examples above or general airline practice.

New customer message: "{new_message}"

Draft reply:"""
    return prompt

def draft_reply(new_message, k=3, similarity_threshold=0.65):
    retrieved = retrieve_similar(new_message, k=k)
    # Filter out weak matches below the threshold - don't ground on irrelevant precedent
    retrieved = [r for r in retrieved if r['score'] >= similarity_threshold]

    if not retrieved:
        return None, "No sufficiently similar historical precedent found (all matches below similarity threshold)."

    prompt = build_reply_prompt(new_message, retrieved)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,  # slight creativity for natural-sounding replies, still mostly grounded
        max_tokens=300,
        reasoning_effort="low",
    )
    return response.choices[0].message.content.strip(), retrieved

In [ ]:
# Cell 54: Test the full reply drafter
test_message = "My flight got delayed 3 hours and I want a refund"
reply, sources = draft_reply(test_message)

print("DRAFTED REPLY:\n")
print(reply)
print("\n\nGROUNDED ON:")
if sources:
    for s in sources:
        print(f"- (sim {s['score']:.2f}) {s['thread'][0]['text'][:100]}...")

DRAFTED REPLY:

@CustomerName We’re sorry to hear about the delay. Please DM us your record locator so we can review your ticket and discuss a refund. Thank you for your patience.


GROUNDED ON:
- (sim 0.89) my flight is now delayed until tomorrow at 9:30am 

@americanair i want a refund...
- (sim 0.68) @AmericanAir it’s ridiculous that after spending hundreds of dollars on a ticket they reufuse to iss...
- (sim 0.68) Our flight delay due to plane maintenance went from 3 hr to 24 hr. Customer service told us there wa...


In [ ]:
# Cell 55: Escalation decision logic (fixed keyword coverage)
ESCALATION_RULES = {
    # intent -> default policy, based on real accuracy data + product risk reasoning
    'flight_disruption_refund': 'escalate',       # money/compensation involved, high stakes
    'service_quality_complaint': 'escalate',      # complaints often seek acknowledgment/compensation
    'non_actionable_other': 'escalate',           # proven unreliable (4-11% accuracy across V1/V2) - escalate out of caution
    'booking_reservation': 'auto_handle',
    'baggage_fees': 'auto_handle',
    'policy_information': 'auto_handle',
    'technical_app_issue': 'auto_handle',
    'positive_feedback': 'auto_handle',           # no real action needed, safe to auto-close
}

def decide_escalation(intent_label, message_text):
    base_decision = ESCALATION_RULES.get(intent_label, 'escalate')  # unknown intent -> escalate by default (safe fallback)

    # Override: certain strong signals always escalate regardless of intent
    escalation_keywords = [
        'lawsuit', 'lawyer', 'sue', 'legal action', 'legal',
        'discriminat', 'injury', 'injured', 'unsafe', 'emergency', 'attorney'
    ]
    text_lower = message_text.lower()
    matched_keywords = [kw for kw in escalation_keywords if kw in text_lower]
    if matched_keywords:
        return 'escalate', f"Escalation keyword override: matched {matched_keywords}"

    reason_map = {
        'flight_disruption_refund': 'Refund/compensation requests carry financial and policy risk requiring human judgment.',
        'service_quality_complaint': 'Service complaints often need acknowledgment or compensation decisions beyond a scripted reply.',
        'non_actionable_other': 'This category showed only 4-11% classification reliability in evaluation; escalating out of caution rather than risking a misrouted real complaint.',
        'booking_reservation': 'Routine informational request, low risk to auto-handle.',
        'baggage_fees': 'Routine informational request, low risk to auto-handle.',
        'policy_information': 'General policy question, safe to answer directly.',
        'technical_app_issue': 'Can be auto-acknowledged; deeper technical issues get escalated by support tooling downstream.',
        'positive_feedback': 'No action needed beyond acknowledgment.',
    }
    reason = reason_map.get(intent_label, 'Unknown/unclassified intent - defaulting to human review.')
    return base_decision, reason

In [ ]:
# Cell 56: Test the escalation logic end-to-end
test_cases = [
    ("flight_disruption_refund", "My flight got delayed 3 hours and I want a refund"),
    ("policy_information", "Can I bring a blanket as carry-on?"),
    ("booking_reservation", "Your staff was so rude I'm considering legal action"),  # mismatched intent, isolates the override
]

for intent, msg in test_cases:
    decision, reason = decide_escalation(intent, msg)
    print(f"Message: {msg}")
    print(f"Intent: {intent} -> Decision: {decision}")
    print(f"Reason: {reason}\n")

Message: My flight got delayed 3 hours and I want a refund
Intent: flight_disruption_refund -> Decision: escalate
Reason: Refund/compensation requests carry financial and policy risk requiring human judgment.

Message: Can I bring a blanket as carry-on?
Intent: policy_information -> Decision: auto_handle
Reason: General policy question, safe to answer directly.

Message: Your staff was so rude I'm considering legal action
Intent: booking_reservation -> Decision: escalate
Reason: Escalation keyword override: matched ['legal action', 'legal']



In [ ]:
# Cell 57: Generate replies for a sample of real customer messages
import random
random.seed(99)  # new seed, distinct sample from anything used before
reply_test_sample = random.sample(resolvable_threads, 40)

reply_eval_data = []
for thread in reply_test_sample:
    customer_msg = thread[0]['text']
    reply, sources = draft_reply(customer_msg, k=3)
    reply_eval_data.append({
        'root_tweet_id': thread[0]['tweet_id'],
        'customer_message': customer_msg,
        'actual_brand_reply': thread[1]['text'] if len(thread) > 1 else None,  # what the brand really said, for reference
        'drafted_reply': reply if reply else "[NO REPLY - insufficient similar precedent]",
    })
    time.sleep(0.3)

reply_eval_df = pd.DataFrame(reply_eval_data)
reply_eval_df.to_csv('/content/drive/MyDrive/hiver_assignment/reply_eval_sample.csv', index=False)
print(f"Generated {len(reply_eval_df)} draft replies")
print(reply_eval_df[['customer_message', 'drafted_reply']].head(3))

Generated 40 draft replies
                                    customer_message  \
0  @AmericanAir I'm hoping you're going to reimbu...   
1  @AmericanAir I’m 5 segments short of achieving...   
2  @348111 We've received your DM and we'll respo...   

                                       drafted_reply  
0  @YourName We understand how frustrating this c...  
1  @YourHandle Thanks for reaching out! To help y...  
2  Thanks for reaching out! Please DM us your rec...  


In [ ]:
# Cell 58: Judge rubric and scoring prompt
JUDGE_RUBRIC = """
Score the DRAFTED REPLY on a 1-5 scale for each dimension:

1. Groundedness (1-5): Does the reply avoid inventing specific facts (amounts, policies, flight details)
   not supported by general airline practice? 5 = fully grounded, no fabrication. 1 = invents false specifics.
2. Relevance (1-5): Does the reply actually address the customer's stated issue? 5 = directly on-point. 1 = off-topic.
3. Tone (1-5): Is the tone appropriately empathetic and professional for a support context? 5 = excellent tone. 1 = dismissive/robotic/inappropriate.
4. Actionability (1-5): Does the reply give the customer a clear next step (if needed)? 5 = clear next step. 1 = vague, no path forward.
"""

def build_judge_prompt(customer_message, drafted_reply):
    return f"""You are evaluating the quality of an AI-drafted customer support reply.

{JUDGE_RUBRIC}

Customer message: "{customer_message}"
Drafted reply: "{drafted_reply}"

Respond in EXACTLY this format, one line per dimension, numbers only:
groundedness: X
relevance: X
tone: X
actionability: X"""

def judge_reply(customer_message, drafted_reply):
    prompt = build_judge_prompt(customer_message, drafted_reply)
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=100,
            reasoning_effort="low",
        )
        text = response.choices[0].message.content.strip()
        scores = {}
        for line in text.split('\n'):
            if ':' in line:
                key, val = line.split(':', 1)
                try:
                    scores[key.strip().lower()] = int(val.strip())
                except ValueError:
                    pass
        return scores, text
    except Exception as e:
        return {}, str(e)

In [ ]:
# Cell 59: Run the judge on all 40 drafted replies
judge_results = []
for _, row in reply_eval_df.iterrows():
    scores, raw = judge_reply(row['customer_message'], row['drafted_reply'])
    judge_results.append({
        'root_tweet_id': row['root_tweet_id'],
        **scores,
        'raw_judge_output': raw,
    })
    time.sleep(0.3)

judge_df = pd.DataFrame(judge_results)
judge_df.to_csv('/content/drive/MyDrive/hiver_assignment/judge_scores.csv', index=False)

print(judge_df[['groundedness', 'relevance', 'tone', 'actionability']].describe())

       groundedness  relevance       tone  actionability
count     36.000000  35.000000  35.000000      31.000000
mean       4.972222   4.685714   4.771429       4.741935
std        0.166667   0.718308   0.490241       0.444803
min        4.000000   2.000000   3.000000       4.000000
25%        5.000000   5.000000   5.000000       4.500000
50%        5.000000   5.000000   5.000000       5.000000
75%        5.000000   5.000000   5.000000       5.000000
max        5.000000   5.000000   5.000000       5.000000


In [ ]:
#run this tomorrow
# Day 2, Cell 1: Full environment restore from Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install -q groq sentence-transformers faiss-cpu tqdm

import os, pickle, time, random
import pandas as pd
import numpy as np
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer

DRIVE_DIR = '/content/drive/MyDrive/hiver_assignment'

# 1. Restore your reconstructed conversation threads
with open(f'{DRIVE_DIR}/unique_threads.pkl', 'rb') as f:
    unique_threads = pickle.load(f)
print(f"Restored {len(unique_threads)} threads")

# 2. Re-establish Groq client
# Cell 18: Set your Groq API key
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass("Paste your Groq API key: ")

from groq import Groq
client = Groq()  # automatically reads GROQ_API_KEY from the environment

# 3. Redefine core functions and taxonomy (cheap - just definitions, no computation)
def format_thread(thread):
    lines = []
    for turn in thread:
        speaker = "Customer" if turn['inbound'] else "Brand"
        lines.append(f"{speaker}: {turn['text']}")
    return "\n".join(lines)

INTENT_TAXONOMY = {
    "flight_disruption_refund": "Delays, cancellations, missed connections, or requests for refunds/compensation due to a disrupted flight.",
    "booking_reservation": "Issues with booking confirmations, reservation references, name changes, or reservation modifications.",
    "baggage_fees": "Questions or complaints about baggage handling, checked-bag fees, or other ancillary fees.",
    "policy_information": "General questions about airline policy, rules, or pricing that are not tied to a specific disrupted trip (e.g., carry-on rules, upgrade eligibility).",
    "service_quality_complaint": "Complaints about staff behavior, rudeness, poor treatment, or onboard discomfort, often seeking acknowledgment or compensation.",
    "technical_app_issue": "Problems with the airline's app, website, or digital check-in/booking systems.",
    "positive_feedback": "Compliments, gratitude, or positive comments with no actionable request.",
    "non_actionable_other": "Sarcasm, off-topic content, out-of-scope requests, or abusive/toxic language not requiring a substantive support response.",
}

def has_substantive_reply(thread):
    brand_turns = [t for t in thread if not t['inbound']]
    return any(len(t['text'].split()) >= 8 for t in brand_turns)

resolvable_threads = [t for t in unique_threads if has_substantive_reply(t)]
print(f"Resolvable threads: {len(resolvable_threads)}")

# 4. Rebuild the embedding index (fast - a minute or two, not worth saving/loading vs recomputing)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
customer_openers = [thread[0]['text'] for thread in resolvable_threads]
embeddings = embed_model.encode(customer_openers, show_progress_bar=True, batch_size=64)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype('float32'))
print(f"FAISS index rebuilt: {index.ntotal} vectors")

def retrieve_similar(query_text, k=3):
    query_vec = embed_model.encode([query_text])
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec.astype('float32'), k)
    return [{'score': float(s), 'thread': resolvable_threads[i]} for s, i in zip(scores[0], indices[0])]

# 5. Restore your already-generated reply eval data (no need to regenerate - saves API calls)
reply_eval_df = pd.read_csv(f'{DRIVE_DIR}/reply_eval_sample.csv')
print(f"Restored {len(reply_eval_df)} pre-generated draft replies")

# 6. Restore yesterday's judge scores too, so Cell 59b's investigation works immediately
judge_df = pd.read_csv(f'{DRIVE_DIR}/judge_scores.csv')
print(f"Restored {len(judge_df)} judge score rows")

print("\n--- Environment restored. Ready to continue from Cell 59b (investigate parsing failures). ---")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.2 MB/s eta 0:00:00
Restored 10428 threads
Resolvable threads: 10286


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

FAISS index rebuilt: 10286 vectors
Restored 40 pre-generated draft replies
Restored 40 judge score rows

--- Environment restored. Ready to continue from Cell 59b (investigate parsing failures). ---


In [ ]:
# Cell 59b: Inspect rows where actionability failed to parse
missing_actionability = judge_df[judge_df['actionability'].isna()]
print(f"Rows with missing actionability: {len(missing_actionability)}")
for _, row in missing_actionability.iterrows():
    print("\n--- Raw judge output ---")
    print(repr(row['raw_judge_output']))

Rows with missing actionability: 9

--- Raw judge output ---
'groundedness: 5\nrelevance: 3\ntone: 5\naction'

--- Raw judge output ---
nan

--- Raw judge output ---
'groundedness: 5\nrelevance: 5\ntone: 5\nactionability:'

--- Raw judge output ---
nan

--- Raw judge output ---
nan

--- Raw judge output ---
'groundedness: 5\nrelevance: 2\ntone: 4'

--- Raw judge output ---
'groundedness: 5\nrelevance: 5\ntone: 5\nactionability:'

--- Raw judge output ---
nan

--- Raw judge output ---
'groundedness: 5'


In [ ]:
# Cell 58b: Fixed judge - more tokens + reasoning control + robust parsing
def judge_reply_v2(customer_message, drafted_reply):
    prompt = build_judge_prompt(customer_message, drafted_reply)
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=250,           # raised further - 100 was clearly too tight
            reasoning_effort="low",
        )
        text = response.choices[0].message.content.strip()
        scores = {}
        for line in text.split('\n'):
            line = line.strip().lstrip('-*• ')
            if ':' in line:
                key, val = line.split(':', 1)
                key = key.strip().lower()
                digits = ''.join(c for c in val if c.isdigit())
                if digits:
                    scores[key] = int(digits[:1])
        return scores, text
    except Exception as e:
        return {}, str(e)

In [ ]:
# Cell 59c: Re-judge all 40 replies with the fix
judge_results_v2 = []
for _, row in reply_eval_df.iterrows():
    scores, raw = judge_reply_v2(row['customer_message'], row['drafted_reply'])
    judge_results_v2.append({'root_tweet_id': row['root_tweet_id'], **scores, 'raw_judge_output': raw})
    time.sleep(0.3)

judge_df = pd.DataFrame(judge_results_v2)
judge_df.to_csv('/content/drive/MyDrive/hiver_assignment/judge_scores.csv', index=False)

print("Missing values per column:")
print(judge_df[['groundedness', 'relevance', 'tone', 'actionability']].isna().sum())
print("\nSummary stats:")
print(judge_df[['groundedness', 'relevance', 'tone', 'actionability']].describe())

Missing values per column:
groundedness     0
relevance        0
tone             0
actionability    0
dtype: int64

Summary stats:
       groundedness  relevance       tone  actionability
count     40.000000  40.000000  40.000000      40.000000
mean       4.800000   4.575000   4.700000       4.600000
std        0.790975   0.902631   0.516398       0.708918
min        1.000000   1.000000   3.000000       2.000000
25%        5.000000   4.750000   4.000000       4.000000
50%        5.000000   5.000000   5.000000       5.000000
75%        5.000000   5.000000   5.000000       5.000000
max        5.000000   5.000000   5.000000       5.000000


In [ ]:
# Cell 60: Blind sample for human validation
random.seed(11)
human_judge_sample = reply_eval_df.sample(n=15, random_state=11)[['root_tweet_id', 'customer_message', 'drafted_reply']].reset_index(drop=True)
for col in ['human_groundedness', 'human_relevance', 'human_tone', 'human_actionability']:
    human_judge_sample[col] = None
human_judge_sample.to_csv('/content/drive/MyDrive/hiver_assignment/human_judge_validation.csv', index=False)
print(f"Prepared {len(human_judge_sample)} pairs to rate")

Prepared 15 pairs to rate


In [ ]:
# Cell 61: Human rating loop - batches of 3, 4 scores each
h_df = pd.read_csv('/content/drive/MyDrive/hiver_assignment/human_judge_validation.csv')
h_df['human_groundedness'] = h_df['human_groundedness'].astype('object')

unlabeled = h_df[h_df['human_groundedness'].isna()].index.tolist()
BATCH = 3

for start in range(0, len(unlabeled), BATCH):
    idx_batch = unlabeled[start:start+BATCH]
    for pos, i in enumerate(idx_batch):
        print(f"\n[{pos+1}] Customer: {h_df.loc[i,'customer_message']}")
        print(f"Drafted reply: {h_df.loc[i,'drafted_reply']}")
    print("\nRate each on 1-5: groundedness,relevance,tone,actionability")
    print(f"Enter {len(idx_batch)} sets, separated by ';' e.g. '5,5,4,5;4,5,5,4;...'")
    raw = input("Ratings: ").strip()
    sets = [s.split(',') for s in raw.split(';')]
    for i, s in zip(idx_batch, sets):
        h_df.at[i,'human_groundedness'] = int(s[0])
        h_df.at[i,'human_relevance'] = int(s[1])
        h_df.at[i,'human_tone'] = int(s[2])
        h_df.at[i,'human_actionability'] = int(s[3])
    h_df.to_csv('/content/drive/MyDrive/hiver_assignment/human_judge_validation.csv', index=False)
    print(f"Saved. Progress: {start+len(idx_batch)}/{len(unlabeled)}")
print("Done.")


[1] Customer: @213819 @AmericanAir It's just the principle, the people at american here were nasty
Drafted reply: @213819 I’m really sorry to hear about your experience. Please DM us your record locator or flight details so we can look into this and make it right. We appreciate your feedback and want to help.

[2] Customer: @AmericanAir disappoints again. My SO missed his connection b/c had no gate and AA wouldn’t wait &lt;10 min. Next flight is in 8 hrs. Great.
Drafted reply: @AmericanAir We’re sorry to hear about the missed connection. Please DM us your reservation number or flight details so we can help re‑book the next available flight and explore any assistance options. We’re here to help.

[3] Customer: .@AmericanAir not making me happy today....
Drafted reply: @117546 We’re sorry to hear that. Please DM us your flight details or record locator so we can look into this right away.

Rate each on 1-5: groundedness,relevance,tone,actionability
Enter 3 sets, separated by ';' e.g. '5

In [ ]:
# Cell 62 (fixed): Merge human and judge scores, compute agreement
h_df = pd.read_csv('/content/drive/MyDrive/hiver_assignment/human_judge_validation.csv')
merged = h_df.merge(judge_df, on='root_tweet_id')

print(f"Matched rows: {len(merged)}\n")
print("Columns available:", merged.columns.tolist())  # sanity check before computing

for dim in ['groundedness', 'relevance', 'tone', 'actionability']:
    diff = (merged[f'human_{dim}'] - merged[dim]).abs()
    exact_match = (diff == 0).mean()
    within_1 = (diff <= 1).mean()
    mean_diff = diff.mean()
    print(f"{dim:15s}: exact match {exact_match:.0%} | within ±1 point {within_1:.0%} | mean abs diff {mean_diff:.2f}")

overall_within_1 = pd.concat([
    (merged[f'human_{d}'] - merged[d]).abs() <= 1 for d in ['groundedness','relevance','tone','actionability']
]).mean()
print(f"\nOverall agreement (within ±1, all dimensions pooled): {overall_within_1:.0%}")

Matched rows: 15

Columns available: ['root_tweet_id', 'customer_message', 'drafted_reply', 'human_groundedness', 'human_relevance', 'human_tone', 'human_actionability', 'groundedness', 'relevance', 'tone', 'actionability', 'raw_judge_output']
groundedness   : exact match 73% | within ±1 point 93% | mean abs diff 0.33
relevance      : exact match 27% | within ±1 point 87% | mean abs diff 0.93
tone           : exact match 40% | within ±1 point 100% | mean abs diff 0.60
actionability  : exact match 73% | within ±1 point 93% | mean abs diff 0.40

Overall agreement (within ±1, all dimensions pooled): 93%


In [ ]:
# Cell 63: Look at where relevance disagreement is biggest
merged['relevance_diff'] = merged['human_relevance'] - merged['relevance']
print(merged[['customer_message', 'drafted_reply', 'human_relevance', 'relevance', 'relevance_diff']].sort_values('relevance_diff', key=abs, ascending=False))

                                     customer_message  \
7   Off to #lax this morning on @americanair for #...   
13  @AmericanAir FL1280 delayed 35 mins at last mi...   
0   @213819 @AmericanAir It's just the principle, ...   
4   Sitting at @AmericanAir gate with 3 hour delay...   
6   @AmericanAir Why do you fly overweight? If we ...   
1   @AmericanAir disappoints again. My SO missed h...   
2         .@AmericanAir not making me happy today....   
14  @AmericanAir Flying you today so get me home g...   
12  @AmericanAir I need to change a flight after d...   
9   @411972 @AmericanAir No, we have to wait to se...   
8   Yesterday @116450 my two bags cost $35 cnd (YV...   
5   Didn’t realize boarding group 9 was a thing un...   
3   @AmericanAir what’s the procedure when my flig...   
10  u lost my luggage im in prague with a hawaiin ...   
11  @AmericanAir - just how long will u make passe...   

                                        drafted_reply  human_relevance  \
7   @336839 T

This table shows something very clear: it's not random noise, it's a systematic bias — and there's one clear outlier that's a different kind of problem entirely.

The pattern: judge is systematically more generous 

Look at 9 of the 15 rows: human said 4, judge said 5 (diff = -1). One case: human said 3, judge said 5 (diff = -2). That's a consistent direction — the judge rounds up toward "perfect relevance" more often than a human would, on replies that are reasonable but not flawless. This is real evidence of leniency bias, not just scatter — worth stating plainly in  report rather than softening it.

The one real outlier: row 7 (diff = +3, human=5, judge=2)

This is the opposite direction and much bigger — worth understanding specifically:

In [ ]:
# Cell 64: Inspect the row 7 anomaly directly
anomaly = merged[merged['relevance_diff'] == merged['relevance_diff'].max()]
print("Customer message:", anomaly['customer_message'].values[0])
print("\nDrafted reply:", anomaly['drafted_reply'].values[0])
print("\nRaw judge output:", anomaly['raw_judge_output'].values[0])

Customer message: Off to #lax this morning on @americanair for #luxuryconnect in #beverlyhills

Drafted reply: @336839 Thanks for flying with us, Nicole! We’re thrilled you’re enjoying Luxury Connect. If there’s anything else we can do to make your trip even better, just DM us your record locator. Safe travels! ✈️🌴

Raw judge output: groundedness: 5
relevance: 2
tone: 4
actionability: 2


Failure mode: Reply drafter over-applies issue-resolution phrasing ("DM your record
locator") to positive/no-ask messages. Example: a customer's happy travel-update tweet
received a reply thanking her but also requesting record-locator info she has no reason
to provide, since nothing needs resolving. Root cause: the retrieval-grounded generation
pool is dominated by complaint/resolution examples, so even a positive-message retrieval
inherits resolution-oriented phrasing. This validates the judge's low relevance/
actionability scores here — closer inspection showed my own human rating (5) had
missed the mismatch on a first read, illustrating that even human evaluation benefits
from a structured rubric rather than a holistic "sounds fine" impression.